# ESCI Classifier Training
**RetailTalk Thesis — ESCI Classification**

Steps:
1. Mount Google Drive & install dependencies
2. Upload CSVs from `finalescidataset/`
3. Prepare dataset (BERT embeddings)
4. Train ESCI classifier
5. Download trained model

> Make sure **Runtime > Change runtime type > GPU (T4)** is selected before running.

## 1. Mount Google Drive (optional — for saving checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Change this path if you want checkpoints saved to your Drive
MODEL_OUTPUT_DIR = '/content/drive/MyDrive/esci_trained_model'
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
print(f'Model will be saved to: {MODEL_OUTPUT_DIR}')

## 2. Install dependencies

In [ ]:
!pip install transformers==4.40.0 scikit-learn seaborn tqdm -q

## 3. Upload CSV files
Upload all 4 CSV files from `finalescidataset/`:
- `esci_dataset_1M_plus.csv`
- `esci_dataset_500k_hard_irrelevant.csv`
- `esci_grocery_1M.csv`
- `esci_dataset_220k_humanized.csv`

In [ ]:
import os
os.makedirs('/content/finalescidataset', exist_ok=True)

from google.colab import files
uploaded = files.upload()

for fname in uploaded:
    os.rename(fname, f'/content/finalescidataset/{fname}')
    print(f'Moved: {fname}')

print('\nFiles in /content/finalescidataset:')
for f in os.listdir('/content/finalescidataset'):
    print(f'  {f}')

## 4. Define model code

In [ ]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup, BertModel, BertTokenizer
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from torch.nn.utils import clip_grad_norm_
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import time
import glob
import pandas as pd
from tqdm.notebook import tqdm
from collections import Counter

ESCI_LABEL_NAMES = ['Exact', 'Substitute', 'Complement', 'Irrelevant']
LABEL_MAP = {'E': 0, 'S': 1, 'C': 2, 'I': 3}
CSV_COLUMNS = ['query', 'product_title', 'esci_label', 'humanized_query']

print('Imports done.')

In [ ]:
class QueryProductClassifier(nn.Module):
    def __init__(self, size_petrained=768, dense_hidden_dim=256, num_dense_layers=2, num_labels=4, dropout_rate=0.1):
        super().__init__()
        self.num_labels = 1 if num_labels <= 2 else num_labels
        self.size_petrained = size_petrained * 2
        fc_layers = []
        prev_dim = self.size_petrained
        self.dropout_embedding = nn.Dropout(dropout_rate)
        for _ in range(num_dense_layers):
            fc_layers.append(nn.Linear(prev_dim, dense_hidden_dim, bias=True))
            fc_layers.append(nn.BatchNorm1d(dense_hidden_dim))
            fc_layers.append(nn.ReLU())
            fc_layers.append(nn.Dropout(dropout_rate))
            prev_dim = dense_hidden_dim
        fc_layers.append(nn.Linear(prev_dim, self.num_labels))
        self.fc = nn.Sequential(*fc_layers)

    def forward(self, query_embedding, product_embedding):
        embedding = torch.cat((query_embedding, product_embedding), 1)
        embedding = self.dropout_embedding(embedding)
        return self.fc(embedding).squeeze(-1)


def generate_dataset(query_embedding, product_embedding, Y):
    return TensorDataset(
        torch.tensor(query_embedding).float(),
        torch.tensor(product_embedding).float(),
        torch.tensor(Y),
    )


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print('Model classes defined.')

## 5. Load and combine CSVs

In [ ]:
CSV_DIR = '/content/finalescidataset'

paths = sorted(glob.glob(os.path.join(CSV_DIR, '*.csv')))
print(f'Found {len(paths)} CSV files:')
dfs = []
for p in paths:
    df = pd.read_csv(p, header=None, names=CSV_COLUMNS)
    print(f'  {os.path.basename(p)}: {len(df):,} rows')
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
combined = combined.dropna(subset=['query', 'product_title', 'esci_label'])
combined = combined[combined['esci_label'].isin(LABEL_MAP)].reset_index(drop=True)

print(f'\nCombined total: {len(combined):,} rows')
print('Label distribution:')
print(combined['esci_label'].value_counts())

## 6. Compute BERT embeddings
This is the slow step — ~1-2 hours for 2.7M rows on GPU T4.

In [ ]:
PREPARED_DIR = '/content/prepared'
os.makedirs(PREPARED_DIR, exist_ok=True)

OUT_QUERIES  = os.path.join(PREPARED_DIR, 'queries.npy')
OUT_PRODUCTS = os.path.join(PREPARED_DIR, 'products.npy')
OUT_LABELS   = os.path.join(PREPARED_DIR, 'labels.npy')

BERT_MODEL_NAME = 'bert-base-multilingual-uncased'
MAX_LENGTH = 32
EMBED_BATCH_SIZE = 512  # larger batch = faster on GPU

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = BertModel.from_pretrained(BERT_MODEL_NAME).to(device)
bert_model.eval()
print('BERT loaded.')

In [ ]:
def embed_texts(texts, tokenizer, model, max_length, batch_size, device):
    all_embs = []
    for start in tqdm(range(0, len(texts), batch_size), desc='Encoding'):
        batch = texts[start:start+batch_size]
        enc = tokenizer(batch, padding='max_length', truncation=True,
                        max_length=max_length, return_tensors='pt')
        with torch.no_grad():
            hidden = model(
                input_ids=enc['input_ids'].to(device),
                attention_mask=enc['attention_mask'].to(device),
                token_type_ids=enc['token_type_ids'].to(device),
            )[0]  # (B, seq_len, 768)
        pooled = F.max_pool1d(hidden.permute(0,2,1), kernel_size=hidden.size(1)).squeeze(-1)
        all_embs.append(pooled.cpu().numpy())
    return np.vstack(all_embs)


if not os.path.exists(OUT_QUERIES):
    print('Encoding queries...')
    q_embs = embed_texts(combined['query'].tolist(), tokenizer, bert_model, MAX_LENGTH, EMBED_BATCH_SIZE, device)
    np.save(OUT_QUERIES, q_embs)
    print(f'Saved queries: {q_embs.shape}')
else:
    print(f'Queries already exist, skipping.')

if not os.path.exists(OUT_PRODUCTS):
    print('Encoding product titles...')
    p_embs = embed_texts(combined['product_title'].tolist(), tokenizer, bert_model, MAX_LENGTH, EMBED_BATCH_SIZE, device)
    np.save(OUT_PRODUCTS, p_embs)
    print(f'Saved products: {p_embs.shape}')
else:
    print(f'Products already exist, skipping.')

if not os.path.exists(OUT_LABELS):
    labels = combined['esci_label'].map(LABEL_MAP).to_numpy(dtype=np.int64)
    np.save(OUT_LABELS, labels)
    print(f'Saved labels: {labels.shape}')
else:
    print(f'Labels already exist, skipping.')

# Free BERT from GPU memory before training
del bert_model
torch.cuda.empty_cache()
print('BERT unloaded from GPU.')

## 7. Train ESCI Classifier

In [ ]:
# --- Config ---
BATCH_SIZE       = 256
NUM_EPOCHS       = 5
LR               = 5e-5
WEIGHT_DECAY     = 0.01
VALIDATION_STEPS = 250
NUM_DEV_EXAMPLES = 5505
RANDOM_STATE     = 42
USE_CLASS_WEIGHTS = True

# Load arrays
query_array  = np.load(OUT_QUERIES)
asin_array   = np.load(OUT_PRODUCTS)
labels_array = np.load(OUT_LABELS)
print(f'Loaded — queries: {query_array.shape}, products: {asin_array.shape}, labels: {labels_array.shape}')

# Train/dev split
dev_size = NUM_DEV_EXAMPLES / len(labels_array)
ids_train, ids_dev = train_test_split(range(len(labels_array)), test_size=dev_size, random_state=RANDOM_STATE)

train_dataset = generate_dataset(query_array[ids_train], asin_array[ids_train], labels_array[ids_train])
dev_dataset   = generate_dataset(query_array[ids_dev],   asin_array[ids_dev],   labels_array[ids_dev])
print(f'Train: {len(train_dataset):,}  Dev: {len(dev_dataset):,}')

In [ ]:
def compute_class_weights(labels_array, num_classes):
    counts = Counter(labels_array.tolist())
    total = len(labels_array)
    weights = [total / (num_classes * counts.get(i, 1)) for i in range(num_classes)]
    min_w = min(weights)
    return [w / min_w for w in weights]


num_labels = 4
set_seed(RANDOM_STATE)

model = QueryProductClassifier(num_labels=num_labels)
model.to(device)

train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=BATCH_SIZE)
val_dataloader   = DataLoader(dev_dataset,   sampler=SequentialSampler(dev_dataset), batch_size=BATCH_SIZE)

total_steps = len(train_dataloader) * NUM_EPOCHS
optimizer   = torch.optim.Adam(model.parameters(), lr=LR, eps=1e-8, weight_decay=WEIGHT_DECAY)
scheduler   = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

if USE_CLASS_WEIGHTS:
    cw = compute_class_weights(labels_array[ids_train], num_labels)
    print(f'Class weights: {cw}')
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device))
else:
    criterion = nn.CrossEntropyLoss()

print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Total training steps: {total_steps:,}')

In [ ]:
best_metric   = 0.0
global_step   = 0
start_epoch   = 0
val_metric_arr = np.empty(len(val_dataloader))
val_loss_arr   = np.empty(len(val_dataloader))
step_start     = time.time()

# Resume from checkpoint if exists
existing_ckpts = sorted([f for f in os.listdir(MODEL_OUTPUT_DIR)
                         if f.startswith('checkpoint_epoch_') and f.endswith('.pt')])
if existing_ckpts:
    ckpt_path = os.path.join(MODEL_OUTPUT_DIR, existing_ckpts[-1])
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch  = ckpt['epoch'] + 1
    global_step  = ckpt['global_step']
    best_metric  = ckpt['best_metric_value']
    print(f'Resumed from {ckpt_path} — epoch {start_epoch}, best metric {best_metric:.4f}')

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f'\n=== Epoch {epoch+1}/{NUM_EPOCHS} ===')

    for batch_idx, batch in enumerate(train_dataloader):
        model.train()
        labels  = batch[2].to(device)
        optimizer.zero_grad()
        logits  = model(batch[0].to(device), batch[1].to(device))
        loss    = criterion(logits.view(-1, num_labels), labels.view(-1))
        preds   = np.argmax(logits.detach().cpu().numpy(), axis=1)
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        global_step += 1

        if batch_idx % VALIDATION_STEPS == 0:
            elapsed   = time.time() - step_start
            s_per_it  = elapsed / global_step if global_step > 0 else 0
            eta_min   = s_per_it * (total_steps - global_step) / 60
            train_acc = accuracy_score(labels.detach().cpu().numpy(), preds)
            print(f'  Train  Ep{epoch+1} Step{global_step}/{total_steps} Loss:{loss:.3f} Acc:{train_acc:.3f} ETA:{eta_min:.1f}min')

            model.eval()
            for vi, vbatch in enumerate(val_dataloader):
                vlabels = vbatch[2].to(device)
                with torch.no_grad():
                    vlogits = model(vbatch[0].to(device), vbatch[1].to(device))
                vloss   = criterion(vlogits.view(-1, num_labels), vlabels.view(-1))
                vpreds  = np.argmax(vlogits.detach().cpu().numpy(), axis=1)
                val_metric_arr[vi] = accuracy_score(vlabels.detach().cpu().numpy(), vpreds)
                val_loss_arr[vi]   = vloss.item()
            val_acc = np.mean(val_metric_arr)
            print(f'  Val    Ep{epoch+1} Step{global_step}/{total_steps} Loss:{np.mean(val_loss_arr):.3f} Acc:{val_acc:.3f}')

            if val_acc > best_metric:
                best_metric = val_acc
                torch.save(model.state_dict(), os.path.join(MODEL_OUTPUT_DIR, 'pytorch_model.bin'))
                print(f'  >> Best model saved (acc={best_metric:.4f})')

    # Save epoch checkpoint
    new_ckpt = os.path.join(MODEL_OUTPUT_DIR, f'checkpoint_epoch_{epoch+1}.pt')
    torch.save({
        'epoch': epoch,
        'global_step': global_step,
        'best_metric_value': best_metric,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
    }, new_ckpt)
    # Remove previous epoch checkpoint
    if epoch > start_epoch:
        prev = os.path.join(MODEL_OUTPUT_DIR, f'checkpoint_epoch_{epoch}.pt')
        if os.path.exists(prev):
            os.remove(prev)
    print(f'Epoch {epoch+1} checkpoint saved: {new_ckpt}')

print(f'\nTraining complete. Best validation accuracy: {best_metric:.4f}')

## 8. Final Evaluation

In [ ]:
best_model_path = os.path.join(MODEL_OUTPUT_DIR, 'pytorch_model.bin')
best_model = QueryProductClassifier(num_labels=4)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.to(device)
best_model.eval()

all_true, all_pred = [], []
for batch in DataLoader(dev_dataset, sampler=SequentialSampler(dev_dataset), batch_size=BATCH_SIZE):
    with torch.no_grad():
        logits = best_model(batch[0].to(device), batch[1].to(device))
    all_true.extend(batch[2].numpy())
    all_pred.extend(np.argmax(logits.cpu().numpy(), axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(classification_report(all_true, all_pred, target_names=ESCI_LABEL_NAMES))

cm = confusion_matrix(all_true, all_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=ESCI_LABEL_NAMES, yticklabels=ESCI_LABEL_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=ESCI_LABEL_NAMES, yticklabels=ESCI_LABEL_NAMES, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalised)')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 9. Download trained model

In [ ]:
from google.colab import files
files.download(os.path.join(MODEL_OUTPUT_DIR, 'pytorch_model.bin'))